In [17]:
# pandas for data manipulation
# re for regular expressions
import re

# pyplot for plotting
import matplotlib.pyplot as plt

# numpy for numerical operations
import numpy as np
import pandas as pd

# other .py files
from utils.csv_import import get_csv_files_generalistic, sort_meta_info

In [18]:
# Retrieve CSV files

path_to: str = "C:\\Users\\pedro\\OneDrive - Universidade de Coimbra\\Ambiente de Trabalho\\tese\\thesis-project"
path: str = path_to + "\\new_data"

FileMap = dict[str, dict[str, str]]
csi_map = dict[str, dict[str, np.ndarray]]


data_files = get_csv_files_generalistic(path)
users_id, activities_id, places_id, esps_id = sort_meta_info(path)

In [19]:
def process_csi(data_file):

	print("data_file:", data_file)
	file_csv = pd.read_csv(data_file, header=None)

	# number of samples
	csi_raw: pd.Series = file_csv.iloc[:, 26]

	total_sc: int = 128
	valid_csi: list[list[float]] = []

    # contar CSI inválidos
	no_match_count: int = 0
	no_complete_count: int = 0

	for entry in csi_raw:
		match = re.search(r"\[(.*?)\]", str(entry))
		if not match:
			no_match_count += 1
			continue
		nums = [float(n) for n in re.findall(r"-?\d+", match.group(1))]
		if len(nums) == total_sc:
			valid_csi.append(nums)
		else:
			no_complete_count += 1

	valid_csi = np.array(valid_csi)

    # (n_amostras, 128)
	complex_csi = valid_csi[:, ::2] + 1j * valid_csi[:, 1::2]

    # coloca sc DC no centro (index 32)
	fft_csi = np.fft.fftshift(complex_csi, axes=1)

    # Extract the 52 active subcarriers (IEEE 802.11n standard): [6:58]
    # (n_amostras, 52)
	active_sc = fft_csi[:, 6:58]

    # Remove subcarriers at positions 25, 26, 27 (center)
    # (n_amostras, 49)
	active_sc = np.delete(active_sc, [25, 26, 27], axis=1)

    # seleciona sub_carriers: 2 a 47
	active_sc = active_sc[:, 2:48]

	return active_sc


def process_magnitude(data_files):

	magnitudes = {}

	for user_key, activities_map in data_files.items():
		magnitudes[user_key] = {}
		print("activities_map:", activities_map)

		for activity, places_map in activities_map.items():
			magnitudes[user_key][activity] = {}
			print("places_map:", places_map)

			for place, esps_map in places_map.items():
				magnitudes[user_key][activity][place] = {}
				print("esps_map:", esps_map)

				for esp, file_path in esps_map.items():
					print("file_path:", file_path)
					if file_path is not None:
						magnitudes[user_key][activity][place][esp] = process_csi(file_path)

	return magnitudes

In [20]:
magnitude_data = process_magnitude(data_files)

activities_map: {'activity_0': {'place_1': {'esp_2': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_02_2026-02-25.csv')], 'esp_3': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_03_2026-02-25.csv')]}}}
places_map: {'place_1': {'esp_2': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_02_2026-02-25.csv')], 'esp_3': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_03_2026-02-25.csv')]}}
esps_map: {'esp_2': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_02_2026-02-25.csv')], 'esp_3': [WindowsPath('C:/Users/pedro/OneDrive - Universidade de Coimbra/Ambiente de Trabalho/tese/thesis-project/new_data/00_00_01_03_2026-02-25.c

ValueError: Invalid file path or buffer object type: <class 'list'>